In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib
import os

print("✅ Libraries imported!")

# Load raw data
df = pd.read_csv('../data/marketing_campaign_dataset.csv')
print("Shape:", df.shape)

✅ Libraries imported!
Shape: (200000, 16)


In [2]:
# 1. Clean Acquisition_Cost (remove $ and commas)
df['Acquisition_Cost'] = df['Acquisition_Cost'].str.replace('$', '').str.replace(',', '').astype(float)

# 2. Extract numeric Duration (e.g., "30 days" → 30)
df['Duration_days'] = df['Duration'].str.extract('(\d+)').astype(int)

print("✅ Data cleaned!")
print("New column Duration_days created")
print(df[['Acquisition_Cost', 'Duration_days']].head())

<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
C:\Users\vaish\AppData\Local\Temp\ipykernel_26168\2147620553.py:5: SyntaxWarning: invalid escape sequence '\d'
  df['Duration_days'] = df['Duration'].str.extract('(\d+)').astype(int)


✅ Data cleaned!
New column Duration_days created
   Acquisition_Cost  Duration_days
0           16174.0             30
1           11566.0             60
2           10200.0             30
3           12724.0             60
4           16452.0             15


In [3]:
# Classification target: Campaign_Success (1 = successful, 0 = not)
# Business rule: Success if ROI >= 5.0 (above average)
df['Campaign_Success'] = (df['ROI'] >= 5.0).astype(int)

print("✅ Targets created!")
print("Success rate:", df['Campaign_Success'].mean() * 100, "% of campaigns are successful")
print("\nFirst 5 targets:")
print(df[['ROI', 'Campaign_Success']].head())

✅ Targets created!
Success rate: 50.2 % of campaigns are successful

First 5 targets:
    ROI  Campaign_Success
0  6.29                 1
1  5.61                 1
2  7.18                 1
3  5.55                 1
4  6.50                 1


In [4]:
# Features we will use (all except ID, Date, raw Duration, targets)
drop_cols = ['Campaign_ID', 'Duration', 'Date']
X = df.drop(drop_cols + ['ROI', 'Campaign_Success'], axis=1)
y_class = df['Campaign_Success']      # for classification
y_reg = df['ROI']                     # for regression

print("✅ Features ready!")
print("Number of features:", X.shape[1])
print(X.columns.tolist())

✅ Features ready!
Number of features: 13
['Company', 'Campaign_Type', 'Target_Audience', 'Channel_Used', 'Conversion_Rate', 'Acquisition_Cost', 'Location', 'Language', 'Clicks', 'Impressions', 'Engagement_Score', 'Customer_Segment', 'Duration_days']


In [5]:
# Identify categorical and numerical columns
categorical_cols = ['Company', 'Campaign_Type', 'Target_Audience', 
                   'Channel_Used', 'Location', 'Language', 'Customer_Segment']
numerical_cols = ['Conversion_Rate', 'Acquisition_Cost', 'Clicks', 
                 'Impressions', 'Engagement_Score', 'Duration_days']

# Create preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

# Fit and transform
X_processed = preprocessor.fit_transform(X)

print("✅ Preprocessing completed!")
print("Final shape for TensorFlow:", X_processed.shape)

✅ Preprocessing completed!
Final shape for TensorFlow: (200000, 42)


In [6]:
# Split for BOTH models (same split so we can compare later)
X_train, X_test, y_class_train, y_class_test = train_test_split(
    X_processed, y_class, test_size=0.2, random_state=42)

_, _, y_reg_train, y_reg_test = train_test_split(
    X_processed, y_reg, test_size=0.2, random_state=42)

print("✅ Train-Test split done!")
print("Training samples:", X_train.shape[0])
print("Testing samples (future campaigns):", X_test.shape[0])

✅ Train-Test split done!
Training samples: 160000
Testing samples (future campaigns): 40000


In [7]:
# Create folder if not exists
os.makedirs('../data/processed', exist_ok=True)

# Save preprocessed data
np.save('../data/processed/X_train.npy', X_train)
np.save('../data/processed/X_test.npy', X_test)
np.save('../data/processed/y_class_train.npy', y_class_train)
np.save('../data/processed/y_class_test.npy', y_class_test)
np.save('../data/processed/y_reg_train.npy', y_reg_train)
np.save('../data/processed/y_reg_test.npy', y_reg_test)

# Save preprocessor for later use
joblib.dump(preprocessor, '../data/processed/preprocessor.joblib')

print("✅ All files saved successfully!")
print("You can now use these in TensorFlow models!")

✅ All files saved successfully!
You can now use these in TensorFlow models!
